# Step 8: Tune Physics Weight (λ)

## Why

Total loss = **data MSE + λ × PDE residual**.

With `λ = 0.01` the physics term was tiny in training.
This notebook sweeps λ and writes `physics_best.pkl` so `02_pinn_model.ipynb` picks it up.

## What “good” looks like

- Too small λ → physics ignored (old behaviour)
- Too large λ → data fit collapses, val MAE gets worse
- Best λ → lowest **val_loss** (data + physics) without blowing up MAE

In [ ]:
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt

from tune_physics import tune

print("X_train exists:", os.path.exists("X_train.npy"))
print("advection.pkl exists:", os.path.exists("advection.pkl"))

## Run the sweep

Short runs (25 epochs) per weight — enough to rank λ, not final training.

In [ ]:
results = tune(weights=[0.0, 0.01, 0.05, 0.1, 0.3], epochs=25)
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(results["physics_weight"], results["best_val_loss"], "o-")
axes[0].set_xlabel("physics_weight (λ)")
axes[0].set_ylabel("best val_loss")
axes[0].set_title("Validation loss vs λ")
axes[0].grid(True, alpha=0.3)

axes[1].plot(results["physics_weight"], results["best_val_mae"], "o-", color="orange")
axes[1].set_xlabel("physics_weight (λ)")
axes[1].set_ylabel("best val_mae (norm)")
axes[1].set_title("Validation MAE vs λ")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
with open("physics_best.pkl", "rb") as f:
    best = pickle.load(f)
print("Use this in 02_pinn_model.ipynb:")
best

## Next

1. Re-run **`02_pinn_model.ipynb`** (loads `physics_best.pkl` automatically)
2. Re-run **`07_improve_forecasts.ipynb`** Parts B–D if you want fresh recal + forecast head
3. Open **`06_evaluation.ipynb`** for the final report